# 04 Phase retrieval — image series / hysteresis loop

Run phase retrieval for every `+`/`-` pair produced by
`01_FTH_series.ipynb`. The support mask, centered pixel mask, phase-retrieval
recipe, CDI propagation/phase/mode, and CDI ROI are reused from the calibrated
single-pair workflow.

In [ ]:
import os, sys
from os.path import join

import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = os.path.abspath(os.getcwd())
sys.path.insert(0, join(BASEFOLDER, "library"))

import importlib
import fthcore as fth
import fth_phase_workflow as wf
wf = importlib.reload(wf)  # Refresh helpers in an existing kernel.
import phase_retrieval_core_unified as pr

print("Base folder:", BASEFOLDER)

## Configuration

In [ ]:
USER = "rb"
SERIES_H5 = join(BASEFOLDER, "processed", "Logs", f"data_recon_series_{USER}.hdf5")
OUTPUT_H5 = join(BASEFOLDER, "processed", "Logs", f"data_phase_retrieval_series_{USER}.hdf5")
RETRIEVED_TYPE = "retrieved_full"  # retrieved_full, retrieved_pc, retrieved_gradient
CLOSE_ALL_EVERY = 5  # Also closes figures created internally by retrieval.
# True passes the saved mask_pixel to phase retrieval; masked detector pixels
# are excluded from the image constraint. False uses every detector pixel.
USE_MASK_PIXEL = True

# Used only if the calibration file was saved before running 04_phase_retrieval.ipynb.
DEFAULT_RECIPE = {
    "algorithm_list": ["HAPRE", "ER", "ER"],
    "number_iterations": [100, 50, 50],
    "helicity": ["+", "+", "-"],
    "beta_zero": 0.5,
    "beta_mode": ["arctan", "const", "const"],
    "alpha_zero": 0.0,
    "alpha_mode": "const",
    "RL_its": [0, 0, 0],
    "RL_freqs": [1e9, 1e9, 1e9],
    "TV_freqs": 1e9,
    "plot_every": 50,
    "average_img": 10,
    "Fourier_last": True,
    "Startimage": [None, "+", "+"],
    "Startgamma": [None, None, None],
    "hologram_intensity_cutoff_vmin": 0.01,
    "hologram_offset": {"+": 0.0, "-": 0.0},
    "output": [False, True, True],
    "modes": [1],
    "normalize_startimage_between_holograms": True,
    "return_format": "auto",
    "crop": 100,
}

## Load and validate the shared calibration

In [ ]:
data = wf.load_data_dict(SERIES_H5)
series = data["series"]
if not series:
    raise ValueError("The FTH series contains no entries")
if data.get("supportmask") is None:
    raise ValueError("No supportmask found. Run 03_define_supportmask.ipynb before 01_FTH_series.ipynb")

supportmask = np.asarray(data["supportmask"])
stored_mask_pixel = np.asarray(data["mask_pixel"], dtype=np.uint8)
mask_pixel = (
    stored_mask_pixel if USE_MASK_PIXEL
    else np.zeros_like(stored_mask_pixel, dtype=np.uint8)
)
experimental_setup = dict(data["experimental_setup"])
recipe = data.get("phase_retrieval_recipe") or DEFAULT_RECIPE
focus_cdi = dict(data.get("focus_cdi", {}))

phase_cdi = float(focus_cdi.get("phase", 0))
prop_dist_cdi = float(focus_cdi.get("prop_dist", 0))
focus_mode_cdi = int(focus_cdi.get("mode", 0))
dx_cdi = float(focus_cdi.get("dx", 0))
dy_cdi = float(focus_cdi.get("dy", 0))
roi_cdi = np.asarray(focus_cdi.get("roi", [0, supportmask.shape[0], 0, supportmask.shape[1]]), dtype=int)

crop = int(recipe["crop"])
crop_shape = np.asarray(supportmask.shape) - 2 * crop
if np.any(crop_shape <= 0):
    raise ValueError(f"Crop {crop} is too large for support shape {supportmask.shape}")
scale = crop_shape / np.asarray(supportmask.shape)
roi_crop = np.rint(roi_cdi * [scale[0], scale[0], scale[1], scale[1]]).astype(int)
roi_crop[[0, 1]] = np.clip(roi_crop[[0, 1]], 0, crop_shape[0])
roi_crop[[2, 3]] = np.clip(roi_crop[[2, 3]], 0, crop_shape[1])
roi_crop_s = wf.roi_to_slices(roi_crop)

print("Entries:", len(series), "support:", supportmask.shape, "retrieved shape:", tuple(crop_shape))
print("CDI focus: prop_dist=", prop_dist_cdi, "phase=", phase_cdi, "mode=", focus_mode_cdi)

## Retrieve and reconstruct every point

In [ ]:
def select_mode(reconstruction, mode):
    modes = wf.as_modes(reconstruction)
    return modes[min(int(mode), modes.shape[0] - 1)]


for index, entry in enumerate(series):
    series_label = f"im_id={entry['holo']['+']['id']},{entry['holo']['-']['id']} | {index + 1}/{len(series)}"
    entry["label"] = series_label
    holograms = {label: np.asarray(entry["holo"][label]["image_c"]) for label in ("+", "-")}
    entry_recipe = dict(recipe)
    entry_recipe["hologram_offset"] = {"+": float(entry.get("offset", 0.0)), "-": 0.0}
    result = pr.phase_retrieval_algorithm(holograms, mask_pixel, supportmask, entry_recipe)

    retrieved = {
        "retrieved_full": result["full_coherence"],
        "retrieved_pc": result["partial_coherence"],
        "retrieved_gradient": result["gradient_descent"],
    }
    for label in ("+", "-"):
        for name, collection in retrieved.items():
            entry["holo"][label][name] = collection.get(label)
        entry["holo"][label]["bsmask"] = result["bsmasks"].get(label)
        entry["holo"][label]["gamma"] = result["gamma"].get(label)

    plus_field = np.asarray(entry["holo"]["+"][RETRIEVED_TYPE])
    minus_field = np.asarray(entry["holo"]["-"][RETRIEVED_TYPE])
    retrieved_shape = wf.spatial_shape(plus_field)
    if retrieved_shape != tuple(crop_shape):
        raise ValueError(f"Point {index}: retrieved shape {retrieved_shape} != expected {tuple(crop_shape)}")
    mask_bs_cdi = np.ones(retrieved_shape)
    plus_recon = select_mode(
        wf.reconstruct_cdi_modes(
            plus_field, mask_bs_cdi, fth, experimental_setup,
            prop_dist=prop_dist_cdi, phase=phase_cdi, dx=dx_cdi, dy=dy_cdi,
        ),
        focus_mode_cdi,
    )
    minus_recon = select_mode(
        wf.reconstruct_cdi_modes(
            minus_field, mask_bs_cdi, fth, experimental_setup,
            prop_dist=prop_dist_cdi, phase=phase_cdi, dx=dx_cdi, dy=dy_cdi,
        ),
        focus_mode_cdi,
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        reconstruction = np.log(plus_recon) - np.log(minus_recon)
    support_eff = wf.resize_binary_to_shape(supportmask, retrieved_shape)
    entry["recon_cdi"] = wf.apply_spatial_mask(
        wf.spatial_roi(reconstruction, roi_crop_s), support_eff[roi_crop_s]
    )
    entry["phase_retrieval_errors"] = {
        "steps": [
            {
                "step": step["step"], "helicity": step["helicity"],
                "mode": step["mode"], "Nit": step["Nit"],
                "RL_it": step["RL_it"], "RL_freq": step["RL_freq"],
                "coherence": step["coherence"], "output": step["output"],
                "error": np.asarray(step["error"]),
                "support_error": np.asarray(step["support_error"]),
            }
            for step in result["error"]["steps"]
        ]
    }

    im_id = int(entry["holo"]["+"]["id"])
    png = join(BASEFOLDER, "processed", f"PhR_series_{index:04d}_ImId_{im_id:04d}_{USER}.png")
    fig, ax = plt.subplots(figsize=(5, 5))
    shown = np.real(entry["recon_cdi"])
    vmin, vmax = wf.finite_percentile_limits(shown)
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    ax.set_title(series_label)
    ax.set_axis_off()
    fig.savefig(png, bbox_inches="tight", dpi=200)
    plt.close(fig)
    entry["phase_retrieval_png"] = png

    output = dict(data)
    output["workflow"] = "phase_retrieval_series"
    output["series"] = series
    output["phase_retrieval_recipe"] = recipe
    output["focus_cdi"] = {
        **focus_cdi, "phase": phase_cdi, "prop_dist": prop_dist_cdi,
        "mode": focus_mode_cdi, "roi": roi_cdi, "roi_crop": roi_crop,
        "retrieved_type": RETRIEVED_TYPE,
    }
    # Checkpoint after each expensive retrieval.
    wf.save_data_dict(output, OUTPUT_H5, overwrite=True)
    print("Retrieved", series_label)
    if CLOSE_ALL_EVERY and (index + 1) % CLOSE_ALL_EVERY == 0:
        plt.close("all")

print("Phase-retrieval series HDF5:", OUTPUT_H5)

## Inspect the reconstructed loop

In [ ]:
stack = np.stack([entry["recon_cdi"] for entry in series])
print("CDI reconstruction stack:", stack.shape)

columns = min(4, len(series))
rows = int(np.ceil(len(series) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows), squeeze=False)
for ax, entry in zip(axes.flat, series):
    shown = np.real(entry["recon_cdi"])
    vmin, vmax = wf.finite_percentile_limits(shown)
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    ax.set_title(entry["label"])
    ax.set_axis_off()
for ax in axes.flat[len(series):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()
plt.close("all")